In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="08-ad-click-prediction",
    notebook_name="01_ad_click_system_design.ipynb"
)

# 01 -- Ad Click Prediction: Complete System Design

**Goal:** Design a system that predicts whether a user will click on an ad.

---

## The Lemonade Stand Analogy

Imagine you have a lemonade stand. Every day, hundreds of people walk by.
Some are thirsty joggers, some are busy commuters, some are kids coming home from school.

You want to hold up a sign that says **"ICE COLD LEMONADE!"** but you can only
hold it up a few times (it's tiring!). So you need to predict:

> **"If I show this sign to THIS person at THIS moment, will they stop and buy?"**

That's EXACTLY what ad click prediction does -- but at the scale of billions of
people and millions of ads, every single day.

**Why does this matter?**
- Google makes over $200B/year from ads
- Meta makes 97% of revenue from ads
- Even a 0.1% improvement = billions of dollars

## 1. Problem Definition

### What We're Predicting

$$P(\text{click} \mid \text{ad}, \text{user}, \text{context})$$

In plain English: "Given this ad, this user, and this context (time, device, etc.),
what is the probability they will click?"

### Business Objective → ML Objective

| Business Goal | ML Translation |
|--------------|----------------|
| Maximize revenue | Predict click probability accurately |
| Show relevant ads | Rank ads by predicted P(click) |
| Keep users happy | Minimize irrelevant ads (low hide rate) |

### Why This Is a Ranking Problem (Pointwise LTR)

We don't compare ads head-to-head. Instead, we score each ad independently
and then sort by score. This is called **pointwise Learning-to-Rank**.

```
Input:  (user_features, ad_features, context_features)
Output: P(click) ∈ [0, 1]
Task:   Binary classification (click = 1, no click = 0)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Let's visualize what the system does at a high level
# Imagine 5 candidate ads for a user
np.random.seed(42)

ads = {
    'ad_id': ['Nike Shoes', 'Car Insurance', 'Python Course', 'Pizza Delivery', 'Travel Deal'],
    'predicted_ctr': [0.08, 0.02, 0.15, 0.05, 0.12],
    'bid_price': [1.50, 3.00, 0.80, 0.50, 2.00],
}

df = pd.DataFrame(ads)
df['expected_revenue'] = df['predicted_ctr'] * df['bid_price']
df = df.sort_values('expected_revenue', ascending=False)

print("Ad Ranking by Expected Revenue (pCTR * bid):")
print("=" * 65)
print(df.to_string(index=False))
print("\nThe top ad gets shown to the user!")
print(f"Winner: {df.iloc[0]['ad_id']} with expected revenue ${df.iloc[0]['expected_revenue']:.3f}")

## 2. Metrics

### The Lemonade Stand Version

How do you know if your sign-holding strategy is working?

- **CTR (Click-Through Rate):** Out of 100 people you showed the sign to, how many stopped? (= clicks / impressions)
- **Cross-Entropy:** How confident and CORRECT were your predictions? If you said "90% chance this jogger buys" and they did, great! If they didn't, that's a big penalty.
- **Revenue:** Did you actually sell more lemonade?

### Offline Metrics (Before Deploying)

In [ ]:
import numpy as np

def cross_entropy(y_true, y_pred, epsilon=1e-7):
    """
    Binary Cross-Entropy Loss.
    
    Think of it like this:
    - If the true label is 1 (clicked) and you predicted 0.9, small penalty
    - If the true label is 1 (clicked) and you predicted 0.1, HUGE penalty
    - If the true label is 0 (not clicked) and you predicted 0.1, small penalty
    - If the true label is 0 (not clicked) and you predicted 0.9, HUGE penalty
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)  # avoid log(0)
    ce = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return ce


def normalized_cross_entropy(y_true, y_pred):
    """
    NCE = CE(model) / CE(baseline)
    
    The baseline always predicts the average CTR.
    NCE < 1 means our model is BETTER than just guessing the average.
    NCE >= 1 means our model is WORSE than just guessing the average (yikes!).
    """
    model_ce = cross_entropy(y_true, y_pred)
    
    # Baseline: always predict the average CTR
    avg_ctr = np.mean(y_true)
    baseline_pred = np.full_like(y_pred, avg_ctr)
    baseline_ce = cross_entropy(y_true, baseline_pred)
    
    return model_ce / baseline_ce


# Example: Compare a good model vs a bad model
y_true = np.array([1, 0, 1, 0, 1, 0, 0, 0, 1, 0])  # 40% CTR

# Good model: high confidence when correct
good_preds = np.array([0.85, 0.10, 0.90, 0.15, 0.80, 0.20, 0.05, 0.10, 0.75, 0.25])

# Bad model: basically random
bad_preds = np.array([0.50, 0.45, 0.55, 0.48, 0.52, 0.47, 0.49, 0.51, 0.53, 0.46])

print("OFFLINE METRICS COMPARISON")
print("=" * 50)
print(f"\nBackground CTR (average): {np.mean(y_true):.1%}")
print(f"\nGood Model:")
print(f"  CE  = {cross_entropy(y_true, good_preds):.4f}")
print(f"  NCE = {normalized_cross_entropy(y_true, good_preds):.4f} (< 1 = good!)")
print(f"\nBad Model:")
print(f"  CE  = {cross_entropy(y_true, bad_preds):.4f}")
print(f"  NCE = {normalized_cross_entropy(y_true, bad_preds):.4f} (close to 1 = bad!)")

In [ ]:
# Visualize: What does cross-entropy actually penalize?
import matplotlib.pyplot as plt
import numpy as np

p = np.linspace(0.01, 0.99, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# When true label = 1 (clicked)
loss_when_click = -np.log(p)
axes[0].plot(p, loss_when_click, 'b-', linewidth=2)
axes[0].set_xlabel('Predicted P(click)', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss when user CLICKED (y=1)', fontsize=14)
axes[0].annotate('Predicted 0.1 for a click\n= HUGE penalty!', 
                 xy=(0.1, -np.log(0.1)), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='red'),
                 xytext=(0.4, 3.0), color='red')

# When true label = 0 (not clicked)
loss_when_no_click = -np.log(1 - p)
axes[1].plot(p, loss_when_no_click, 'r-', linewidth=2)
axes[1].set_xlabel('Predicted P(click)', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Loss when user DID NOT click (y=0)', fontsize=14)
axes[1].annotate('Predicted 0.9 for a non-click\n= HUGE penalty!', 
                 xy=(0.9, -np.log(0.1)), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='red'),
                 xytext=(0.3, 3.0), color='red')

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 5)

plt.tight_layout()
plt.savefig('cross_entropy_intuition.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nKey insight: CE heavily penalizes CONFIDENT WRONG predictions.")
print("That is why calibration matters so much in ad systems!")

### Online Metrics (After Deploying)

| Metric | Formula | Lemonade Stand Version |
|--------|---------|----------------------|
| **CTR** | clicked / shown | How many people who saw your sign actually bought lemonade? |
| **Conversion Rate** | conversions / impressions | How many people who clicked actually completed a purchase? |
| **Revenue Lift** | % revenue increase | Are you making more money this week than last? |
| **Hide Rate** | hidden / shown | How many people actively told you to go away? (bad!) |

## 3. High-Level Architecture

The system has 3 pipelines working together:

```
┌─────────────────────────────────────────────────────────────────┐
│                    PREDICTION PIPELINE                         │
│                                                                │
│  User Query                                                    │
│      │                                                         │
│      ▼                                                         │
│  ┌──────────────────────┐                                      │
│  │ Candidate Generation │  ← Uses advertiser targeting         │
│  │ (filter millions of  │    (age, gender, country)            │
│  │  ads → small subset) │                                      │
│  └──────────┬───────────┘                                      │
│             │                                                  │
│             ▼                                                  │
│  ┌──────────────────────┐     ┌───────────────┐               │
│  │   Ranking Service    │◄────│ Feature Store  │               │
│  │ (predict P(click)    │     │ (batch + online│               │
│  │  for each candidate) │     │  features)     │               │
│  └──────────┬───────────┘     └───────────────┘               │
│             │                         ▲                        │
│             ▼                         │                        │
│  ┌──────────────────────┐     ┌───────────────┐               │
│  │  Re-Ranking Service  │     │  DATA PREP    │               │
│  │ (diversity, business │     │  PIPELINE     │               │
│  │  rules, heuristics)  │     │ • Batch feats │               │
│  └──────────┬───────────┘     │ • Online feats│               │
│             │                 │ • Training    │               │
│             ▼                 │   data gen    │               │
│  Top K Ads                    └───────────────┘               │
│                                       ▲                        │
│                               ┌───────────────┐               │
│                               │  CONTINUAL    │               │
│                               │  LEARNING     │               │
│                               │ • Fine-tune   │               │
│                               │ • Evaluate    │               │
│                               │ • Deploy      │               │
│                               └───────────────┘               │
└─────────────────────────────────────────────────────────────────┘
```

## 4. Data Pipeline

### Three Types of Logs

Think of it like a security camera at your lemonade stand:

1. **Impression logs:** Someone walked by and SAW the sign
2. **Click logs:** Someone stopped and picked up the cup
3. **Conversion logs:** Someone actually paid and drank the lemonade

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Simulate real interaction logs
np.random.seed(42)

n_impressions = 1000

# Generate impression logs
impression_logs = pd.DataFrame({
    'impression_id': range(n_impressions),
    'user_id': np.random.randint(1, 100, n_impressions),
    'ad_id': np.random.randint(1, 50, n_impressions),
    'timestamp': pd.date_range('2024-01-01', periods=n_impressions, freq='1min'),
    'dwell_time_sec': np.random.exponential(3, n_impressions).round(1),
    'position': np.random.randint(1, 6, n_impressions),  # Position 1-5 on timeline
})

# Some impressions become clicks (~5% CTR, typical for social media)
click_prob = 0.05
clicked = np.random.random(n_impressions) < click_prob
click_logs = impression_logs[clicked].copy()
click_logs['click_delay_sec'] = np.random.exponential(5, clicked.sum()).round(1)

# Some clicks become conversions (~10% of clicks)
conversion_prob = 0.10
converted = np.random.random(len(click_logs)) < conversion_prob
conversion_logs = click_logs[converted].copy()

print("DATA PIPELINE SUMMARY")
print("=" * 50)
print(f"Impressions: {len(impression_logs):,}")
print(f"Clicks:      {len(click_logs):,} (CTR = {len(click_logs)/len(impression_logs):.1%})")
print(f"Conversions: {len(conversion_logs):,} (CVR = {len(conversion_logs)/len(impression_logs):.2%})")
print(f"\nFunnel: {len(impression_logs)} → {len(click_logs)} → {len(conversion_logs)}")
print(f"\nSample impression log:")
print(impression_logs.head(3).to_string(index=False))

In [ ]:
# Constructing training data from logs
# This is a critical step that interviewers love to ask about!

CLICK_WINDOW_SEC = 30  # If user clicks within 30 seconds → positive
MIN_DWELL_TIME_SEC = 1.0  # Must see ad for at least 1 second to count

def construct_training_data(impression_logs, click_logs, 
                            click_window=CLICK_WINDOW_SEC,
                            min_dwell=MIN_DWELL_TIME_SEC):
    """
    Convert raw logs into training examples.
    
    The Lemonade Stand Version:
    - If someone walked by so fast they couldn't even read your sign → SKIP
    - If someone read the sign and bought lemonade within 30 sec → POSITIVE
    - If someone read the sign but walked away → NEGATIVE
    """
    # Filter: only count impressions where user actually SAW the ad
    valid_impressions = impression_logs[impression_logs['dwell_time_sec'] >= min_dwell].copy()
    
    # Create set of clicked impression IDs
    clicked_ids = set(click_logs[
        click_logs['click_delay_sec'] <= click_window
    ]['impression_id'])
    
    # Label: 1 if clicked within window, 0 otherwise
    valid_impressions['label'] = valid_impressions['impression_id'].isin(clicked_ids).astype(int)
    
    return valid_impressions

training_data = construct_training_data(impression_logs, click_logs)

print("TRAINING DATA CONSTRUCTION")
print("=" * 50)
print(f"Total impressions:   {len(impression_logs):,}")
print(f"Valid impressions:   {len(training_data):,} (dwell >= {MIN_DWELL_TIME_SEC}s)")
print(f"Filtered out:        {len(impression_logs) - len(training_data):,} (too fast to see)")
print(f"\nPositive (clicked):  {training_data['label'].sum():,}")
print(f"Negative (not clicked): {(training_data['label'] == 0).sum():,}")
print(f"Class ratio: 1:{(training_data['label'] == 0).sum() // max(training_data['label'].sum(), 1)}")
print(f"\nNote: This class imbalance is NORMAL in ad systems!")
print(f"Typical CTR is 1-5%, so you get 20-100x more negatives than positives.")

## 5. Feature Engineering

### The Lemonade Stand Version

When deciding who to show your sign to, you'd look at:

- **About the sign (ad features):** Is it colorful? Does it mention price? How many people liked it before?
- **About the person (user features):** Are they a jogger (thirsty)? Have they bought lemonade before? How old are they?
- **About the moment (context features):** Is it hot outside? Is it lunchtime? Are they on a phone or computer?
- **Combinations (cross features):** A jogger on a hot day is WAY more likely to buy than either feature alone!

In [ ]:
import pandas as pd
import numpy as np

# Simulate feature engineering for ad click prediction
np.random.seed(42)

n_samples = 5

# ============================================================
# AD FEATURES
# ============================================================
ad_features = pd.DataFrame({
    'ad_id': [101, 202, 303, 404, 505],
    'advertiser_id': [1, 2, 1, 3, 2],
    'campaign_id': [10, 20, 10, 30, 20],
    'category': ['travel', 'insurance', 'travel', 'food', 'insurance'],
    'subcategory': ['hotel', 'car', 'airline', 'pizza', 'health'],
    'total_impressions': [50000, 12000, 75000, 8000, 30000],
    'total_clicks': [2500, 360, 6000, 640, 1200],
    'historical_ctr': [0.050, 0.030, 0.080, 0.080, 0.040],
})

print("AD FEATURES")
print("=" * 80)
print(ad_features.to_string(index=False))

# ============================================================
# USER FEATURES
# ============================================================
user_features = pd.DataFrame({
    'user_id': [1001, 1002, 1003, 1004, 1005],
    'age': [25, 35, 19, 42, 28],
    'gender': ['M', 'F', 'M', 'F', 'M'],
    'country': ['US', 'UK', 'US', 'CA', 'US'],
    'total_ad_views': [500, 200, 1000, 150, 800],
    'historical_click_rate': [0.03, 0.05, 0.01, 0.08, 0.02],
    'recent_categories_clicked': [
        'travel,food', 'insurance,finance', 'gaming,tech',
        'food,health', 'travel,sports'
    ],
})

print("\nUSER FEATURES")
print("=" * 80)
print(user_features.to_string(index=False))

# ============================================================
# CONTEXT FEATURES
# ============================================================
context_features = pd.DataFrame({
    'device': ['mobile', 'desktop', 'mobile', 'tablet', 'mobile'],
    'hour_of_day': [8, 14, 22, 10, 18],
    'day_of_week': ['Mon', 'Wed', 'Fri', 'Sat', 'Tue'],
    'is_weekend': [False, False, False, True, False],
})

print("\nCONTEXT FEATURES")
print("=" * 50)
print(context_features.to_string(index=False))

In [ ]:
# Why feature interactions matter -- a visual demonstration
import matplotlib.pyplot as plt
import numpy as np

# Scenario: CTR depends on INTERACTION between age group and ad category
categories = ['Travel', 'Insurance', 'Gaming', 'Food']
age_groups = ['18-24', '25-34', '35-44', '45+']

# CTR matrix (age_group x category)
ctr_matrix = np.array([
    [0.04, 0.01, 0.12, 0.06],  # 18-24: love gaming, meh on insurance
    [0.08, 0.05, 0.06, 0.07],  # 25-34: balanced, starting to care about insurance
    [0.06, 0.09, 0.02, 0.05],  # 35-44: insurance is relevant now!
    [0.10, 0.08, 0.01, 0.04],  # 45+: travel and insurance, not gaming
])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(ctr_matrix, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=12)
ax.set_yticks(range(len(age_groups)))
ax.set_yticklabels(age_groups, fontsize=12)

# Add text annotations
for i in range(len(age_groups)):
    for j in range(len(categories)):
        ax.text(j, i, f'{ctr_matrix[i, j]:.1%}', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if ctr_matrix[i, j] > 0.07 else 'black')

ax.set_xlabel('Ad Category', fontsize=14)
ax.set_ylabel('Age Group', fontsize=14)
ax.set_title('CTR Depends on INTERACTION Between Age and Category\n'
             '(This is why feature crosses matter!)', fontsize=14)
plt.colorbar(im, label='Click-Through Rate')
plt.tight_layout()
plt.savefig('feature_interactions.png', dpi=100, bbox_inches='tight')
plt.show()

print("KEY INSIGHT:")
print("A 20-year-old seeing a Gaming ad: 12% CTR")
print("A 20-year-old seeing an Insurance ad: 1% CTR")
print("Neither 'age=20' nor 'category=gaming' alone tells you this!")
print("You need the INTERACTION between them.")

## 6. Model Choices: From Simple to Sophisticated

### The Progression

```
Logistic Regression      → Simple, fast, but linear (can't learn interactions)
    │
    ▼
Feature Cross + LR       → Manual interactions, but requires domain expertise
    │
    ▼
GBDT                     → Good, but can't do continual learning
    │
    ▼
GBDT + LR                → Facebook's classic approach
    │
    ▼
Neural Network           → Powerful, but struggles with sparse features
    │
    ▼
DCN                      → Google's approach: auto feature crosses
    │
    ▼
Factorization Machines   → Elegant pairwise interactions
    │
    ▼
DeepFM                   → Best of both: FM + DNN (recommended!)
```

In [ ]:
# Quick demo: Logistic Regression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

np.random.seed(42)

# Generate synthetic ad click data
n_samples = 10000

# Features: age, hour_of_day, historical_ctr, ad_impressions, is_mobile
X = np.column_stack([
    np.random.normal(30, 10, n_samples),           # age
    np.random.randint(0, 24, n_samples),            # hour
    np.random.beta(2, 50, n_samples),               # user historical CTR
    np.random.exponential(10000, n_samples),         # ad total impressions
    np.random.binomial(1, 0.7, n_samples),           # is_mobile
])

# Create labels with some realistic pattern
# Click probability depends on interactions between features
logits = (
    -3.0                           # base (low CTR)
    + 0.02 * X[:, 0]              # slight age effect
    + 0.5 * X[:, 2] * 20          # historical CTR matters a lot
    + 0.3 * X[:, 4]               # mobile slightly higher CTR
    + np.random.normal(0, 0.5, n_samples)  # noise
)
y = (np.random.random(n_samples) < 1 / (1 + np.exp(-logits))).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train LR
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Evaluate
y_pred_prob = lr.predict_proba(X_test_scaled)[:, 1]

print("LOGISTIC REGRESSION BASELINE")
print("=" * 50)
print(f"AUC-ROC:   {roc_auc_score(y_test, y_pred_prob):.4f}")
print(f"Log Loss:  {log_loss(y_test, y_pred_prob):.4f}")
print(f"Avg CTR:   {y_test.mean():.3f}")
print(f"Avg pCTR:  {y_pred_prob.mean():.3f}")
print(f"\nFeature weights:")
feature_names = ['age', 'hour', 'user_hist_ctr', 'ad_impressions', 'is_mobile']
for name, weight in zip(feature_names, lr.coef_[0]):
    print(f"  {name:20s}: {weight:+.4f}")
print(f"\nNote: LR can only capture LINEAR effects.")
print(f"It CANNOT learn that 'young + gaming = high CTR' automatically.")

In [ ]:
# Comparison: Model capabilities at a glance
import matplotlib.pyplot as plt
import numpy as np

models = ['LR', 'Cross+LR', 'GBDT', 'GBDT+LR', 'NN', 'DCN', 'FM', 'DeepFM']
criteria = [
    'Pairwise\nInteractions',
    'Higher-order\nInteractions',
    'Continual\nLearning',
    'Handles\nSparsity',
    'Training\nSpeed',
]

# Score each model on each criterion (0-5)
scores = np.array([
    [1, 1, 5, 2, 5],   # LR
    [3, 1, 5, 1, 4],   # Cross+LR
    [3, 3, 1, 3, 3],   # GBDT
    [3, 3, 2, 3, 3],   # GBDT+LR
    [3, 5, 4, 2, 2],   # NN
    [4, 4, 4, 3, 2],   # DCN
    [5, 1, 4, 5, 3],   # FM
    [5, 5, 4, 5, 2],   # DeepFM
])

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(scores.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=5)

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(criteria)))
ax.set_yticklabels(criteria, fontsize=11)

for i in range(len(criteria)):
    for j in range(len(models)):
        score = scores[j, i]
        label = ['', 'Poor', 'Fair', 'Good', 'Great', 'Best'][score]
        ax.text(j, i, f'{label}', ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='white' if score <= 1 or score >= 4 else 'black')

ax.set_title('Model Comparison for Ad Click Prediction\n'
             '(Green = Good, Red = Bad)', fontsize=14)
plt.colorbar(im, ticks=[0, 1, 2, 3, 4, 5], label='Score')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nINTERVIEW ANSWER:")
print('"I would start with LR as a baseline, then experiment with')
print(' DCN and DeepFM, as both are widely used in the tech industry."')

## 7. System Diagrams

### Serving Architecture

```
                              ┌─────────────────────┐
                              │   New Interactions   │
                              │ (clicks, impressions)│
                              └──────────┬──────────┘
                                         │
                    ┌────────────────────┬┴───────────────────┐
                    │                    │                    │
                    ▼                    ▼                    ▼
          ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐
          │  Batch Feature  │  │ Online Feature   │  │  Training Data  │
          │  Computation    │  │ Computation      │  │  Generation     │
          │ (every N days)  │  │ (real-time)      │  │  (continuous)   │
          └────────┬────────┘  └────────┬────────┘  └────────┬────────┘
                   │                    │                    │
                   ▼                    ▼                    ▼
          ┌──────────────────────────────────┐     ┌─────────────────┐
          │          Feature Store            │     │  Model Training │
          │  (batch + real-time features)     │     │  (continual)    │
          └──────────────┬───────────────────┘     └────────┬────────┘
                         │                                  │
                         │         ┌────────────────────────┘
                         │         │
                         ▼         ▼
          ┌──────────────────────────────────┐
          │        Prediction Pipeline        │
          │                                    │
          │   User → Candidate Gen → Rank     │
          │          → Re-Rank → Top K Ads    │
          └──────────────────────────────────┘
```

### Batch vs Online Features

| Type | Examples | Update Frequency | Computed |
|------|----------|-----------------|----------|
| **Batch (Static)** | Ad image embedding, ad category, user demographics | Every few days/weeks | Batch job → Feature Store |
| **Online (Dynamic)** | Ad impression count, click count, user's session clicks | Real-time | At query time |

In [ ]:
# Simulate the prediction pipeline
import numpy as np
import time

class AdClickPredictionPipeline:
    """Simplified simulation of the ad click prediction serving pipeline."""
    
    def __init__(self):
        # Simulated ad inventory
        self.all_ads = {
            i: {
                'ad_id': i,
                'category': np.random.choice(['travel', 'insurance', 'gaming', 'food', 'tech']),
                'target_age_min': np.random.randint(18, 35),
                'target_age_max': np.random.randint(35, 65),
                'target_countries': np.random.choice(['US', 'UK', 'CA', 'ALL']),
                'bid': round(np.random.uniform(0.5, 5.0), 2),
                'historical_ctr': round(np.random.beta(2, 50), 4),
            }
            for i in range(1000)  # 1000 ads in inventory
        }
    
    def candidate_generation(self, user):
        """Stage 1: Filter ads by targeting criteria (fast, coarse)."""
        candidates = []
        for ad_id, ad in self.all_ads.items():
            # Check targeting
            if (ad['target_age_min'] <= user['age'] <= ad['target_age_max'] and
                ad['target_countries'] in [user['country'], 'ALL']):
                candidates.append(ad)
        return candidates
    
    def rank(self, user, candidates):
        """Stage 2: Score each candidate with the ML model (slow, precise)."""
        scored = []
        for ad in candidates:
            # Simplified: pCTR based on historical CTR + some user affinity
            affinity = 1.5 if ad['category'] in user.get('interests', []) else 1.0
            pctr = min(ad['historical_ctr'] * affinity * (1 + user.get('click_rate', 0.03)), 1.0)
            expected_revenue = pctr * ad['bid']
            scored.append({**ad, 'pctr': pctr, 'expected_revenue': expected_revenue})
        
        # Sort by expected revenue
        scored.sort(key=lambda x: x['expected_revenue'], reverse=True)
        return scored
    
    def rerank(self, ranked_ads, top_k=5):
        """Stage 3: Apply business rules (diversity, frequency capping)."""
        # Simple diversity: don't show more than 2 ads from same category
        category_count = {}
        final_ads = []
        for ad in ranked_ads:
            cat = ad['category']
            if category_count.get(cat, 0) < 2:
                final_ads.append(ad)
                category_count[cat] = category_count.get(cat, 0) + 1
            if len(final_ads) >= top_k:
                break
        return final_ads
    
    def predict(self, user, top_k=5):
        """Full pipeline: candidate generation → ranking → re-ranking."""
        t0 = time.time()
        
        # Stage 1: Candidate Generation
        candidates = self.candidate_generation(user)
        t1 = time.time()
        
        # Stage 2: Ranking
        ranked = self.rank(user, candidates)
        t2 = time.time()
        
        # Stage 3: Re-ranking
        final = self.rerank(ranked, top_k)
        t3 = time.time()
        
        return {
            'ads': final,
            'stats': {
                'total_ads': len(self.all_ads),
                'candidates_after_filter': len(candidates),
                'final_shown': len(final),
                'candidate_gen_ms': (t1 - t0) * 1000,
                'ranking_ms': (t2 - t1) * 1000,
                'reranking_ms': (t3 - t2) * 1000,
                'total_ms': (t3 - t0) * 1000,
            }
        }


# Run the pipeline!
pipeline = AdClickPredictionPipeline()

user = {
    'user_id': 42,
    'age': 28,
    'country': 'US',
    'interests': ['travel', 'tech'],
    'click_rate': 0.05,
}

result = pipeline.predict(user)

print("AD CLICK PREDICTION PIPELINE")
print("=" * 70)
print(f"\nUser: age={user['age']}, country={user['country']}, interests={user['interests']}")
print(f"\nFunnel: {result['stats']['total_ads']} total ads")
print(f"  → {result['stats']['candidates_after_filter']} after targeting filter")
print(f"  → {result['stats']['final_shown']} shown to user")
print(f"\nLatency: {result['stats']['total_ms']:.1f}ms total")
print(f"  Candidate gen: {result['stats']['candidate_gen_ms']:.1f}ms")
print(f"  Ranking:       {result['stats']['ranking_ms']:.1f}ms")
print(f"  Re-ranking:    {result['stats']['reranking_ms']:.1f}ms")
print(f"\nTop ads shown:")
for i, ad in enumerate(result['ads']):
    print(f"  #{i+1}: ad_{ad['ad_id']} | {ad['category']:10s} | "
          f"pCTR={ad['pctr']:.4f} | bid=${ad['bid']:.2f} | "
          f"E[rev]=${ad['expected_revenue']:.4f}")

## Summary: What to Say in an Interview

### The 2-Minute Version

1. **Problem:** Predict P(click | user, ad, context) to maximize ad revenue
2. **Framing:** Pointwise LTR with binary classification
3. **Data:** Impression/click/conversion logs; label positives within t seconds, negatives otherwise
4. **Features:** Ad features (IDs, category, historical CTR), user features (demographics, click history, engagement stats), context (device, time), and crucially -- feature interactions
5. **Model:** Start with LR baseline, then DeepFM (FM for pairwise interactions + DNN for higher-order)
6. **Metrics:** NCE offline, CTR/revenue lift/hide rate online
7. **Serving:** 3 pipelines -- data prep (batch+online features), continual learning, prediction (candidate gen → rank → re-rank)
8. **Key challenges:** Sparse features, continual learning (5-min delay hurts), calibration for auction

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)